# Flex-search goal comparison on subject ernie

This notebook runs three small studies with `tit.opt.flex` on the SimNIBS example subject `ernie`: (1) a comparison of six optimization goals (`mean`, `focality_tf` at two intensity weights, ROC `focality` with fixed and adaptive thresholds, and `mean` with an optimized current ratio) on two subcortical targets (thalamus, left hippocampus); (2) a focused look at `mean` vs. the current-ratio variant; and (3) a one-factor-at-a-time exploration of the differential-evolution hyperparameters. Study 1+2 is 12 optimizer runs and study 3 adds ~10 more -- at SimNIBS's default population/iteration budget each run takes on the order of an hour, so the whole notebook takes **many hours** end to end.

Every runner cell is resumable: it skips a run whose output already has a `flex_meta.json`, so re-running this notebook (or a single cell) after an interruption picks up where it left off. Run it inside the TI-Toolbox v3.0.0 container with the `ernie` example data reachable (see cell 1).

## 0. Setup

Set `PROJECT` to a folder on your machine (or export `TIT_PROJECT_DIR` before starting the kernel). `fetch_ernie` downloads the example subject's finished head model once.

Set the environment variable `TIT_STUDY_SMOKE=1` before starting the kernel to shrink every run to `max_iterations=2, population_size=4` and prefix output folders with `smoke_` -- that is how this notebook was validated before shipping; leave it unset for the real study.

In [ ]:
import os
import time
import json
import glob
import csv

from tit import get_path_manager
from tit.examples import fetch_ernie

PROJECT = os.environ.get("TIT_PROJECT_DIR") or "/path/to/your/project"  # <-- edit me (or set TIT_PROJECT_DIR)
SUBJECT = "ernie"

SMOKE = os.environ.get("TIT_STUDY_SMOKE") == "1"
OUT_PREFIX = "smoke_" if SMOKE else "study_"

pm = get_path_manager(PROJECT)
fetch_ernie(PROJECT)

ATLAS_PATH = os.path.join(pm.m2m(SUBJECT), "segmentation", "labeling.nii.gz")
STUDY_DIR = pm.flex_search(SUBJECT)
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__" if "__file__" in dir() else "."))

print("project:", PROJECT)
print("atlas:  ", ATLAS_PATH)
print("outputs:", STUDY_DIR)
print("SMOKE = ", SMOKE)

A tiny helper picks the right runner for `config.mode`, the way `tit.opt.flex.__main__` dispatches, and records timing/FEM-evaluation-count rows to a CSV so a study can be audited without re-running it.

In [ ]:
from tit.opt import FlexConfig, run_flex_search
from tit.opt.flex.drivers import run_adaptive_focality


def run_flex(config):
    """Dispatch on config.mode the same way tit.opt.flex.__main__ does."""
    if config.mode == FlexConfig.Mode.FLEX_ADAPTIVE:
        return run_adaptive_focality(config)
    return run_flex_search(config)


def _fem_evaluations_from_log(output_folder):
    """Best-effort FEM-evaluation count parsed from the run's own log/summary.

    SimNIBS writes a line "Total number of FEM evaluations: N" to summary.txt
    inside the numbered run folder (e.g. <output_folder>/00/summary.txt).
    Returns None if no such line is found (e.g. the adaptive driver's
    unnamed step-1 folder is not scanned here).
    """
    for summary_path in sorted(glob.glob(os.path.join(output_folder, "*", "summary.txt"))):
        try:
            text = open(summary_path).read()
        except OSError:
            continue
        for line in text.splitlines():
            if "Total number of FEM evaluations" in line:
                try:
                    return int(line.split(":")[-1].strip())
                except ValueError:
                    pass
    return None


def _already_done(output_folder):
    """Resumable check: only the named study folder's 00/flex_meta.json counts.

    (Known bug: the adaptive driver's step-1 mean pass writes a bare
    timestamped folder directly under flex-search/ -- that folder is never
    what output_folder points at, so it is naturally ignored here.)
    """
    return os.path.exists(os.path.join(output_folder, "00", "flex_meta.json"))


def run_study(configs, timing_csv):
    """Run a list of (name, FlexConfig) pairs sequentially and resumably.

    Appends one row per run to timing_csv (created with a header if new).
    Skips a run whose output_folder already has a completed 00/flex_meta.json.
    """
    is_new = not os.path.exists(timing_csv)
    with open(timing_csv, "a", newline="") as f:
        writer = csv.writer(f)
        if is_new:
            writer.writerow(["name", "start", "end", "exit", "best_value", "fem_evaluations", "output_folder"])
        for name, config in configs:
            if _already_done(config.output_folder):
                print(f"SKIP (already done): {name}")
                continue
            print(f"RUN: {name} -> {config.output_folder}")
            start = time.time()
            exit_code = 0
            best_value = None
            try:
                result = run_flex(config)
                exit_code = 0 if result.success else 1
                best_value = result.best_value
            except Exception as exc:
                exit_code = 1
                print(f"  FAILED: {name}: {exc}")
            end = time.time()
            fem_evals = _fem_evaluations_from_log(config.output_folder)
            writer.writerow([name, start, end, exit_code, best_value, fem_evals, config.output_folder])
            f.flush()
            print(f"DONE: {name} exit={exit_code} elapsed={end - start:.0f}s best_value={best_value} fem_evals={fem_evals}")

## 1. Study 1 -- goal comparison

Two targets (thalamus, left hippocampus) x six arms: `mean`, `focality_tf` with intensity weight 0 and 1, ROC `focality` with a fixed threshold pair, ROC `focality` with adaptive (80/20) thresholds, and `mean` with an optimized current ratio (`mean_ratio` -- this arm is also study 2, see below). Budget: SimNIBS's own DE defaults, left as `None` (`max_iterations`, `population_size`, `mutation`, `recombination`), with a tighter `tolerance=0.01` and `n_multistart=1`. The SimNIBS ROC `thresholds` string convention is `"<nonroi>,<roi>"` (see `tit/opt/flex/pareto.py`), so the fixed-threshold arm below uses `"0.1,0.2"` for a 0.1 V/m non-ROI / 0.2 V/m ROI threshold pair.

In [ ]:
TARGETS = {
    "thalamus": FlexConfig.SubcorticalROI(atlas_path=ATLAS_PATH, label=[10, 49], tissues="GM", atlas_space="subject"),
    "lhippo": FlexConfig.SubcorticalROI(atlas_path=ATLAS_PATH, label=17, tissues="GM", atlas_space="subject"),
}

STUDY1_HYPERPARAMS = dict(n_multistart=1, tolerance=0.01) if not SMOKE else dict(
    n_multistart=1, max_iterations=2, population_size=4, tolerance=None
)


def base_flex_kwargs(target_name, hyperparams):
    return dict(
        subject_id=SUBJECT,
        current_mA=2.0,
        electrode=FlexConfig.ElectrodeConfig(),
        roi=TARGETS[target_name],
        run_final_electrode_simulation=True,
        **hyperparams,
    )


def build_arm(target_name, arm, hyperparams):
    kwargs = base_flex_kwargs(target_name, hyperparams)
    if arm == "mean":
        kwargs.update(goal="mean", postproc="max_TI")
    elif arm == "tf_w0":
        kwargs.update(goal="focality_tf", postproc="max_TI", intensity_weight=0.0)
    elif arm == "tf_w1":
        kwargs.update(goal="focality_tf", postproc="max_TI", intensity_weight=1.0)
    elif arm == "roc_fixed":
        kwargs.update(goal="focality", postproc="max_TI", thresholds="0.1,0.2", non_roi_method="everything_else")
    elif arm == "roc_adapt":
        kwargs.update(
            goal="focality", postproc="max_TI", non_roi_method="everything_else",
            mode="flex_adaptive", adaptive=FlexConfig.AdaptiveFocalityConfig(),
        )
    elif arm == "mean_ratio":
        kwargs.update(goal="mean", postproc="max_TI", optimize_current_ratio=True, ratio_levels=21)
    else:
        raise ValueError(f"unknown arm {arm!r}")
    name = f"{target_name}_{arm}"
    kwargs["output_folder"] = os.path.join(STUDY_DIR, f"{OUT_PREFIX}study_{name}")
    return name, FlexConfig(**kwargs)


ARMS = ["mean", "tf_w0", "tf_w1", "roc_fixed", "roc_adapt", "mean_ratio"]
study1_configs = [
    build_arm(target, arm, STUDY1_HYPERPARAMS)
    for target in ["thalamus", "lhippo"]
    for arm in ARMS
]
[name for name, _ in study1_configs]

In [ ]:
STUDY1_TIMING_CSV = os.path.join(STUDY_DIR, f"{OUT_PREFIX}study1_timing.csv")
run_study(study1_configs, STUDY1_TIMING_CSV)

## 2. Study 1 analysis

For each finished run: load the two final-simulation channel meshes (`final_sim_0/*_TDCS_1_scalar.msh`, `final_sim_1/*_TDCS_1_scalar.msh`), restrict to gray-matter tetrahedra (SimNIBS tissue tag 2), compute the TI envelope on those elements with `tit.calc.get_TI_vectors`, and label each element ROI/non-ROI by looking its centroid up in `labeling.nii.gz` (nearest voxel, via the image's inverse affine). All summary statistics are volume-weighted (tetrahedron volume as the weight) since element size varies across the mesh.

In [ ]:
import numpy as np
import nibabel as nib
import simnibs

from tit.calc import get_TI_vectors
from tit.opt.flex.objectives import threshold_free_focality

GM_TAG = 2
ROI_TARGET_LABELS = {"thalamus": [10, 49], "lhippo": [17]}


def _channel_field(output_folder, channel_index):
    """E-field vectors [V/m] on GM tetrahedra for one final-simulation channel."""
    sim_dir = os.path.join(output_folder, "00", f"final_sim_{channel_index}")
    msh_paths = glob.glob(os.path.join(sim_dir, "*_TDCS_1_scalar.msh"))
    if not msh_paths:
        raise FileNotFoundError(f"no final-sim mesh under {sim_dir}")
    mesh = simnibs.read_msh(msh_paths[0])
    gm = mesh.crop_mesh(tags=[GM_TAG])
    e_field = gm.field["E"].value  # (N, 3) V/m
    centroids = gm.elements_baricenters().value  # (N, 3) mm, subject (native) space
    volumes = gm.elements_volumes_and_areas().value  # (N,) mm^3
    return e_field, centroids, volumes


def _roi_mask_from_atlas(centroids, target_labels):
    img = nib.load(ATLAS_PATH)
    data = img.get_fdata()
    inv_affine = np.linalg.inv(img.affine)
    homog = np.concatenate([centroids, np.ones((centroids.shape[0], 1))], axis=1)
    voxel = (inv_affine @ homog.T).T[:, :3]
    voxel_idx = np.round(voxel).astype(int)
    shape = data.shape
    in_bounds = np.all((voxel_idx >= 0) & (voxel_idx < np.array(shape)), axis=1)
    labels = np.zeros(centroids.shape[0], dtype=data.dtype)
    labels[in_bounds] = data[voxel_idx[in_bounds, 0], voxel_idx[in_bounds, 1], voxel_idx[in_bounds, 2]]
    return np.isin(labels, target_labels)


def _weighted_mean(values, weights):
    return float(np.sum(values * weights) / np.sum(weights))


def _weighted_percentile(values, weights, q):
    order = np.argsort(values)
    v, w = values[order], weights[order]
    cum = np.cumsum(w) - 0.5 * w
    cum /= np.sum(w)
    return float(np.interp(q / 100.0, cum, v))


def _weighted_auc(roi_values, roi_weights, nonroi_values, nonroi_weights):
    """Volume-weighted rank-based (Mann-Whitney) ROI-vs-non-ROI AUC.

    Each element contributes its tetrahedron volume as a weight, i.e. this
    is the probability that a randomly chosen unit of ROI volume has a
    higher field magnitude than a randomly chosen unit of non-ROI volume.
    """
    values = np.concatenate([roi_values, nonroi_values])
    weights = np.concatenate([roi_weights, nonroi_weights])
    is_roi = np.concatenate([np.ones_like(roi_values, dtype=bool), np.zeros_like(nonroi_values, dtype=bool)])
    order = np.argsort(values)
    values, weights, is_roi = values[order], weights[order], is_roi[order]
    total_roi_w = weights[is_roi].sum()
    total_nonroi_w = weights[~is_roi].sum()
    if total_roi_w == 0 or total_nonroi_w == 0:
        return float("nan")
    cum_nonroi_below = np.cumsum(np.where(is_roi, 0.0, weights))
    # weight of non-ROI strictly below each element (ties split evenly by not
    # double counting equal values -- adequate at this element count/precision)
    nonroi_below_roi = cum_nonroi_below[is_roi] - np.where(is_roi, 0.0, weights)[is_roi]
    return float(np.sum(nonroi_below_roi * weights[is_roi]) / (total_roi_w * total_nonroi_w))


def analyze_run(name, output_folder, target):
    meta_path = os.path.join(output_folder, "00", "flex_meta.json")
    if not os.path.exists(meta_path):
        return None
    e0, centroids, volumes = _channel_field(output_folder, 0)
    e1, _, _ = _channel_field(output_folder, 1)
    ti = get_TI_vectors([e0, e1])
    ti_mag = np.linalg.norm(ti, axis=1)

    roi_mask = _roi_mask_from_atlas(centroids, ROI_TARGET_LABELS[target])
    roi_vals, roi_w = ti_mag[roi_mask], volumes[roi_mask]
    nonroi_vals, nonroi_w = ti_mag[~roi_mask], volumes[~roi_mask]

    roi_mean = _weighted_mean(roi_vals, roi_w) if roi_w.sum() > 0 else float("nan")
    nonroi_mean = _weighted_mean(nonroi_vals, nonroi_w) if nonroi_w.sum() > 0 else float("nan")
    nonroi_p95 = _weighted_percentile(nonroi_vals, nonroi_w, 95) if nonroi_w.sum() > 0 else float("nan")
    auc = _weighted_auc(roi_vals, roi_w, nonroi_vals, nonroi_w)
    offtarget_frac = float(np.sum(nonroi_w[nonroi_vals >= 0.2]) / nonroi_w.sum()) if nonroi_w.sum() > 0 else float("nan")
    tf_score = threshold_free_focality(roi_vals, nonroi_vals, intensity_weight=0.0) if roi_vals.size and nonroi_vals.size else float("nan")

    with open(meta_path) as f:
        meta = json.load(f)

    return dict(
        name=name,
        target=target,
        roi_mean=roi_mean,
        nonroi_mean=nonroi_mean,
        focality_ratio=roi_mean / nonroi_mean if nonroi_mean else float("nan"),
        nonroi_p95=nonroi_p95,
        auc=auc,
        offtarget_vol_frac_ge_0p2=offtarget_frac,
        threshold_free_focality_w0=tf_score,
        best_value=meta.get("best_value"),
        n_roi_elements=int(roi_mask.sum()),
        n_nonroi_elements=int((~roi_mask).sum()),
        ti_mag=ti_mag,
        roi_mask=roi_mask,
    )

In [ ]:
import pandas as pd

study1_rows = []
study1_field_cache = {}  # name -> analyze_run(...) dict, kept for the figures below
for name, config in study1_configs:
    target = name.split("_", 1)[0]
    try:
        row = analyze_run(name, config.output_folder, target)
    except FileNotFoundError as exc:
        print(f"skip {name}: {exc}")
        row = None
    if row is None:
        continue
    study1_field_cache[name] = row
    study1_rows.append({k: v for k, v in row.items() if k not in ("ti_mag", "roi_mask")})

study1_df = pd.DataFrame(study1_rows)
study1_results_csv = os.path.join(STUDY_DIR, f"{OUT_PREFIX}study1_results.csv")
study1_df.to_csv(study1_results_csv, index=False)
study1_df

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG_DIR = os.path.join(STUDY_DIR, f"{OUT_PREFIX}study1_figures")
os.makedirs(FIG_DIR, exist_ok=True)


def arm_of(name):
    return name.split("_", 1)[1]


# (a) summary bar/scatter per goal: AUC and ROI mean, one panel per target.
if not study1_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric in zip(axes, ["auc", "roi_mean"]):
        for target in study1_df["target"].unique():
            sub = study1_df[study1_df["target"] == target].copy()
            sub["arm"] = sub["name"].apply(arm_of)
            sub = sub.sort_values("arm")
            ax.plot(sub["arm"], sub[metric], marker="o", label=target)
        ax.set_title(metric)
        ax.tick_params(axis="x", rotation=45)
        ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, "summary_auc_roimean.png"), dpi=120)
    plt.close(fig)

    # (b) ROC curves per target, all goals overlaid: sweep a threshold on the
    # TI envelope and plot (non-ROI volume fraction above threshold) vs.
    # (ROI volume fraction above threshold).
    for target in study1_df["target"].unique():
        fig, ax = plt.subplots(figsize=(5, 5))
        for name, row in study1_field_cache.items():
            if row["target"] != target:
                continue
            ti_mag, roi_mask = row["ti_mag"], row["roi_mask"]
            thresholds = np.linspace(0, np.percentile(ti_mag, 99.5), 60)
            roi_frac, nonroi_frac = [], []
            for thr in thresholds:
                roi_frac.append(np.mean(ti_mag[roi_mask] >= thr) if roi_mask.any() else 0.0)
                nonroi_frac.append(np.mean(ti_mag[~roi_mask] >= thr) if (~roi_mask).any() else 0.0)
            ax.plot(nonroi_frac, roi_frac, label=arm_of(name))
        ax.plot([0, 1], [0, 1], "k--", linewidth=0.5)
        ax.set_xlabel("non-ROI volume fraction above threshold")
        ax.set_ylabel("ROI volume fraction above threshold")
        ax.set_title(f"ROC-style curves -- {target}")
        ax.legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(os.path.join(FIG_DIR, f"roc_{target}.png"), dpi=120)
        plt.close(fig)

    # (c) field-distribution panels per target x goal: ROI vs non-ROI histograms.
    targets = list(study1_df["target"].unique())
    arms_present = sorted(set(arm_of(n) for n in study1_field_cache))
    fig, axes = plt.subplots(len(targets), len(arms_present), figsize=(3 * len(arms_present), 3 * len(targets)), squeeze=False)
    for i, target in enumerate(targets):
        for j, arm in enumerate(arms_present):
            ax = axes[i][j]
            name = f"{target}_{arm}"
            row = study1_field_cache.get(name)
            if row is None:
                ax.axis("off")
                continue
            ti_mag, roi_mask = row["ti_mag"], row["roi_mask"]
            ax.hist(ti_mag[~roi_mask], bins=30, alpha=0.6, label="non-ROI", density=True)
            ax.hist(ti_mag[roi_mask], bins=30, alpha=0.6, label="ROI", density=True)
            if i == 0:
                ax.set_title(arm, fontsize=9)
            if j == 0:
                ax.set_ylabel(target, fontsize=9)
    axes[0][0].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, "field_distributions.png"), dpi=120)
    plt.close(fig)

print("figures written to", FIG_DIR)

## 3. Study 2 -- mean vs. current-ratio optimization

Study 2 is a slice of the study-1 table: the `mean` and `mean_ratio` rows per target. `mean_ratio` additionally optimizes the split of the 2 mA total current between the two electrode pairs; the chosen split is recorded in the run's own `summary.txt`/`flex_meta.json` under the optimizer's current-ratio result (look for `"current_ratio"` / the per-channel currents in `flex_meta.json` -- printed below for each `mean_ratio` run).

In [ ]:
study2_df = study1_df[study1_df["name"].apply(lambda n: arm_of(n) in ("mean", "mean_ratio"))]
display(study2_df)

for name, config in study1_configs:
    if not name.endswith("mean_ratio"):
        continue
    meta_path = os.path.join(config.output_folder, "00", "flex_meta.json")
    if not os.path.exists(meta_path):
        continue
    with open(meta_path) as f:
        meta = json.load(f)
    print(name, "flex_meta.json keys:", sorted(meta.keys()))
    for key in meta:
        if "ratio" in key.lower() or "current" in key.lower():
            print("  ", key, "=", meta[key])

## 4. Study 3 -- hyperparameter exploration

One-factor-at-a-time exploration of the DE hyperparameters around SimNIBS's defaults (`tolerance=0.1`, `mutation=(0.01, 0.5)`, `recombination=0.7`, `population_size=13`), using `focality_tf` (w=0) on the thalamus target only. The exact-default configuration is run 3 times (seed cannot be fixed -- `FlexConfig` has no seed field, so these repeats measure the DE's run-to-run noise at fixed hyperparameters) and reused as the "default" point for every other factor, so it is not re-run per factor.

In [ ]:
def build_hp_config(name, overrides):
    kwargs = base_flex_kwargs("thalamus", dict(
        n_multistart=1,
        max_iterations=1000,
        population_size=13,
        tolerance=0.1,
        mutation="0.01,0.5",
        recombination=0.7,
    ))
    kwargs.update(goal="focality_tf", postproc="max_TI", intensity_weight=0.0)
    kwargs.update(overrides)
    if SMOKE:
        kwargs.update(max_iterations=2, population_size=4)
    kwargs["output_folder"] = os.path.join(STUDY_DIR, f"{OUT_PREFIX}hp_{name}")
    return name, FlexConfig(**kwargs)


study3_configs = [build_hp_config("default_rep0", {})]
if not SMOKE:
    study3_configs += [
        build_hp_config("default_rep1", {}),
        build_hp_config("default_rep2", {}),
        build_hp_config("tolerance_0.01", dict(tolerance=0.01)),
        build_hp_config("tolerance_0.001", dict(tolerance=0.001)),
        build_hp_config("mutation_0.5,1.0", dict(mutation="0.5,1.0")),
        build_hp_config("mutation_0.3,0.9", dict(mutation="0.3,0.9")),
        build_hp_config("recombination_0.5", dict(recombination=0.5)),
        build_hp_config("recombination_0.9", dict(recombination=0.9)),
        build_hp_config("population_8", dict(population_size=8)),
        build_hp_config("population_20", dict(population_size=20)),
    ]
else:
    # SMOKE budget: default + one tolerance variant + one mutation variant.
    study3_configs += [
        build_hp_config("tolerance_0.01", dict(tolerance=0.01)),
        build_hp_config("mutation_0.5,1.0", dict(mutation="0.5,1.0")),
    ]

[name for name, _ in study3_configs]

In [ ]:
STUDY3_TIMING_CSV = os.path.join(STUDY_DIR, f"{OUT_PREFIX}study3_timing.csv")
run_study(study3_configs, STUDY3_TIMING_CSV)

In [ ]:
study3_rows = []
for name, config in study3_configs:
    try:
        row = analyze_run(name, config.output_folder, "thalamus")
    except FileNotFoundError as exc:
        print(f"skip {name}: {exc}")
        continue
    if row is None:
        continue
    row.pop("ti_mag", None)
    row.pop("roi_mask", None)
    study3_rows.append(row)

study3_df = pd.DataFrame(study3_rows)

# Merge in wall time / FEM-evaluation count from the timing CSV.
if os.path.exists(STUDY3_TIMING_CSV):
    timing_df = pd.read_csv(STUDY3_TIMING_CSV)
    timing_df["wall_time_s"] = timing_df["end"] - timing_df["start"]
    study3_df = study3_df.merge(
        timing_df[["name", "wall_time_s", "fem_evaluations"]], on="name", how="left"
    )

study3_results_csv = os.path.join(STUDY_DIR, f"{OUT_PREFIX}study3_results.csv")
study3_df.to_csv(study3_results_csv, index=False)
study3_df

In [ ]:
if not study3_df.empty and "fem_evaluations" in study3_df:
    fig, ax = plt.subplots(figsize=(6, 5))
    default_rows = study3_df[study3_df["name"].str.startswith("default_rep")]
    other_rows = study3_df[~study3_df["name"].str.startswith("default_rep")]
    ax.scatter(other_rows["fem_evaluations"], other_rows["best_value"], label="factor variants")
    ax.scatter(default_rows["fem_evaluations"], default_rows["best_value"], color="red", marker="x", s=80, label="default repeats (seed noise)")
    for _, r in study3_df.iterrows():
        ax.annotate(r["name"], (r["fem_evaluations"], r["best_value"]), fontsize=6)
    ax.set_xlabel("FEM evaluations")
    ax.set_ylabel("best_value (objective)")
    ax.set_title("Study 3: objective vs. cost, default-repeat spread highlighted")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(STUDY_DIR, f"{OUT_PREFIX}study3_objective_vs_fem_evals.png"), dpi=120)
    plt.close(fig)

print("study 3 outputs:", STUDY3_TIMING_CSV, study3_results_csv)

## 5. Where the results are

- `flex-search/study_<target>_<arm>/00/` -- one finished optimizer run per study-1/2 arm (mesh, `flex_meta.json`, `summary.txt`, `final_sim_0/`, `final_sim_1/`).
- `flex-search/hp_<factor>_<value>/00/` and `flex-search/hp_default_rep<k>/00/` -- study-3 runs.
- `flex-search/study1_timing.csv`, `flex-search/study3_timing.csv` -- one row per run: start/end/exit/best_value/FEM-evaluation count, appended as each run finishes (safe to tail while the notebook is still running).
- `flex-search/study1_results.csv`, `flex-search/study3_results.csv` -- the analysis tables built in cells above.
- `flex-search/study1_figures/` -- the goal-comparison PNGs (summary bars, ROC-style curves, field-distribution panels).
- `flex-search/study3_objective_vs_fem_evals.png` -- the hyperparameter-exploration figure.

(When `TIT_STUDY_SMOKE=1`, every one of the paths above is prefixed with `smoke_` instead of `study_`/bare, so a smoke pass never collides with the real study's outputs.)